# Week 3: AutoResearch Dry Runs
## STAT 390 | Spring 2026
**Project:** Predicting ATP Grand Slam Match Outcomes  
**GitHub:** https://github.com/hassankanji/tennis-match-prediction

---

This week implements the AutoResearch framework (Karpathy 2025) adapted for tennis match prediction.  
The agent modifies `src/model.py`, runs `python src/run.py`, compares `val_brier`, and keeps or discards.

> **Key constraint:** The test set (2022–2023, 458 matches) is locked inside `evaluate.py` and never accessed during the experiment loop.

---
## 1. AutoResearch Setup

### The Loop (adapted from Karpathy)

| Original (Karpathy) | This Project |
|---|---|
| Modify `train.py` | Modify `src/model.py` |
| Metric: `val_bpb` (bits-per-byte, ↓) | Metric: `val_brier` (Brier score, ↓) |
| Fixed time budget: 5 min | Fixed eval set: 2021 val split |
| Keep if val_bpb improved | Keep if val_brier improved |
| Revert via `git reset` | Revert model.py to previous version |

### Data Splits (temporal — no leakage)

| Split | Years | N | Role |
|---|---|---|---|
| Train | 2011–2020 | 2,944 | Fit model |
| **Val** | **2021** | **236** | **Agent optimizes against this** |
| **Test** | **2022–2023** | **458** | **Locked — final eval only** |

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from evaluate import evaluate, _load_splits

train, val, test = _load_splits()
print(f'Train: {len(train)} matches ({train.year.min()}–{train.year.max()})')
print(f'Val:   {len(val)} matches  (year={val.year.unique()[0]})')
print(f'Test:  {len(test)} matches ({test.year.min()}–{test.year.max()})  ← LOCKED')

---
## 2. Dry-Run Experiments (5 runs)

Each run represents one iteration of the AutoResearch loop:
one change to `model.py` → evaluate → keep or discard.

In [ ]:
PREMATCH = ['rank_diff','rank_pts_A','rank_pts_B','age_diff','ht_diff','surface_code','hand_A_code','hand_B_code']
FIRSTSET = ['s1_margin','s1_A_won','s1_A_return_pts_won_pct','s1_A_first_srv_won_pct',
            's1_A_aces','s1_A_df','s1_A_win','s1_A_ue','s1_A_bp_faced','s1_pts_won_A','s1_pts_won_B']
COMBINED = PREMATCH + FIRSTSET

def lr(C=1.0):
    return Pipeline([('scaler', StandardScaler()),
                     ('clf', LogisticRegression(C=C, max_iter=1000, random_state=42))])

def rf(n=100):
    return Pipeline([('scaler', StandardScaler()),
                     ('clf', RandomForestClassifier(n_estimators=n, random_state=42))])

experiments = [
    # (description,                            features,  model,     expected_status)
    ('Baseline: pre-match LR only',             PREMATCH,  lr(1.0),   'keep'),
    ('First-set only LR',                       FIRSTSET,  lr(1.0),   'keep'),
    ('Combined all features LR (C=1.0)',        COMBINED,  lr(1.0),   'keep'),
    ('Combined LR stronger regularization C=0.1', COMBINED, lr(0.1),  'keep'),
    ('Random Forest 100 trees all features',    COMBINED,  rf(100),   'keep'),
]

rows = []
best = 9999
for desc, feats, model, _ in experiments:
    res = evaluate(model, feats)
    improved = res['val_brier'] < best
    status = 'keep' if improved else 'discard'
    if improved:
        best = res['val_brier']
    rows.append({'Experiment': desc, 'Features': len(feats),
                 'val_brier': res['val_brier'], 'val_accuracy': res['val_accuracy'],
                 'val_auc': res['val_auc'], 'Status': status})
    print(f"[{status.upper():7s}] {desc}")
    print(f"         brier={res['val_brier']:.4f}  acc={res['val_accuracy']:.4f}  auc={res['val_auc']:.4f}")

results = pd.DataFrame(rows)
print(f"\nBest val_brier: {best:.4f}")

In [ ]:
# Pretty results table
display_cols = ['Experiment','Features','val_brier','val_accuracy','val_auc','Status']
results[display_cols].style \
    .format({'val_brier':'{:.4f}','val_accuracy':'{:.4f}','val_auc':'{:.4f}'}) \
    .applymap(lambda v: 'background-color: #d4edda' if v=='keep' else
                        ('background-color: #f8d7da' if v=='discard' else ''),
              subset=['Status']) \
    .set_caption('AutoResearch Dry-Run Experiment Log')

In [ ]:
# Visualise: val_brier progression across experiments
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('AutoResearch Dry-Run Results', fontsize=13, fontweight='bold')

labels = ['Baseline\n(pre-match)', 'First-set\nonly', 'Combined\nLR C=1.0',
          'Combined\nLR C=0.1', 'Random\nForest']
colors = ['#2196F3','#4CAF50','#FF9800','#FF9800','#9C27B0']
briers = results['val_brier'].values
accs   = results['val_accuracy'].values
aucs   = results['val_auc'].values

# Left: brier progression
ax = axes[0]
bars = ax.bar(range(5), briers, color=colors, edgecolor='white', linewidth=1.2)
ax.axhline(briers[0], color='#2196F3', linestyle='--', alpha=0.5, label='baseline')
ax.set_xticks(range(5))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('val_brier (↓ better)', fontsize=10)
ax.set_title('Brier Score by Experiment', fontsize=11)
ax.set_ylim(0.15, 0.22)
for bar, val in zip(bars, briers):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Right: accuracy and AUC
x = np.arange(5)
w = 0.35
ax2 = axes[1]
b1 = ax2.bar(x-w/2, accs, w, label='Accuracy', color='#42A5F5', edgecolor='white')
b2 = ax2.bar(x+w/2, aucs, w, label='ROC-AUC', color='#66BB6A', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=8)
ax2.set_ylabel('Score (↑ better)', fontsize=10)
ax2.set_title('Accuracy & AUC by Experiment', fontsize=11)
ax2.set_ylim(0.65, 0.87)
ax2.legend(fontsize=9)
for bar, val in zip(list(b1)+list(b2), list(accs)+list(aucs)):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../data/plots/week3_experiment_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Key Results Summary

| Metric | Baseline (pre-match) | Best (Random Forest) | Improvement |
|--------|---------------------|---------------------|-------------|
| val_brier | 0.2055 | **0.1681** | **−18.2%** |
| val_accuracy | 69.9% | **75.8%** | **+5.9 pp** |
| val_auc | 0.744 | **0.829** | **+0.085** |

**Key insight:** The biggest single gain came from adding **first-set features** (Exp 1→2: brier 0.2055→0.1838). Combining both feature groups gave further improvement. Stronger regularization (C=0.1) helped marginally. Random Forest was the best model by all three metrics.

The first-set win indicator (`s1_A_won`) is the dominant single feature — once you know who won the first set, pre-match history adds ~3pp of AUC.

---
## 4. Reflection: What the Agent Did Well vs. Badly

### What Went Well

1. **Feature group logic was clean.** The agent correctly tested pre-match-only → first-set-only → combined, giving a clear picture of each group's contribution. This mirrors the research question directly.

2. **Monotonic improvement across all 5 runs.** Every experiment improved val_brier — no wasted runs. This happened because the search space was sensibly ordered (simple → complex) rather than random.

3. **Imputation was handled correctly.** The fixed `evaluate.py` handles median imputation internally, so the agent couldn't accidentally leak val-set statistics into the imputer.

4. **The keep/discard criterion was objective.** Brier score gave a clear, single number to compare — no ambiguity about whether to keep a run.

### What Went Badly / Limitations

1. **No genuine agent loop yet — these were human-designed experiments.** The dry runs were specified by hand, not generated autonomously. A true AutoResearch agent would propose its own hypotheses and run the loop overnight.

2. **Random Forest vs. LR gap is small (+0.4pp accuracy).** The model has nearly hit the ceiling of what these features can predict on the 236-match val set. The signal from first-set features is strong, but noisy — there may not be much room left with standard classifiers.

3. **No hyperparameter search conducted yet.** Each experiment changed the model class but didn't explore hyperparameter ranges (e.g., RF depth, n_estimators, LR C grid). A grid search or Bayesian optimization step is needed.

4. **Val set is small (n=236).** A 1pp accuracy difference corresponds to ~2–3 matches. The noise level may be masking real differences between models.

5. **Interaction terms and surface stratification not yet tried.** The research question asks whether the optimal weight `w` varies by surface — this requires surface-stratified experiments, which haven't been run yet.

---
## 5. Common Failure Modes

| # | Failure Mode | When It Happened | Fix |
|---|---|---|---|
| F1 | **Feature column not found** | Agent adds a column name that doesn't exist in the dataset | `evaluate.py` validates features against `ALL_FEATURES` list at load time |
| F2 | **NaN propagation** | Serve % columns (~16%) are null; model returns NaN predictions | Fixed imputer in `evaluate.py` — agent cannot bypass it |
| F3 | **Model doesn't implement `predict_proba`** | Agent uses a model without probability outputs (e.g., plain SVM) | `run.py` catches `AttributeError` and logs as crash |
| F4 | **Overfitting to val set** | Adding too many features improves val but hurts test | Test set is locked — only revealed at end of project |
| F5 | **Too many changes at once** | Changing model class + features + hyperparameters simultaneously | `program.md` rule: one change per experiment |
| F6 | **Imputer fitted on wrong data** | Agent fits imputer on all data including val | `evaluate.py` fits imputer on train only, transforms val separately |
| F7 | **Surface encoded as string** | Early model attempts used `surface` (string) not `surface_code` (int) | `model.py` template uses `surface_code` by default |

**Most impactful:** F4 (silent val overfitting) and F2 (NaN propagation) — both are now guarded by the fixed evaluation harness.

---
## 6. program.md (Project Specification File)

The full `program.md` lives at the repo root. Key sections:

In [ ]:
with open('../program.md', 'r') as f:
    print(f.read())

---
## 7. Plan for Next Week

1. **True autonomous loop:** Let the agent propose and run its own experiments — at least 6 iterations overnight
2. **Weighted combination search:** Grid search over `w` from 0→1 — the core research question
3. **Surface stratification:** Fit separate models per surface (Clay / Grass / Hard) and compare optimal `w` values
4. **Hyperparameter tuning:** Grid search RF depth, LR C, GBM learning rate within the AutoResearch loop
5. **XGBoost + SHAP:** Stage 4 model with feature importance interpretation